# PyCaret End-to-End Classification – Banking Loan Default

## Use Case
Build an end-to-end **loan default classification model** using PyCaret.

### Target
- `loan_default = 0` → Customer is **not expected to default**
- `loan_default = 1` → Customer is **expected to default**

### What this notebook demonstrates
1. Load the banking dataset
2. Basic data understanding and EDA
3. Check missing values and class balance
4. Initialize PyCaret
5. Automatically preprocess categorical and numerical columns
6. Compare multiple classification algorithms
7. Select the best model
8. Tune the selected model
9. Evaluate using Accuracy, Precision, Recall, F1 and ROC-AUC
10. Plot Confusion Matrix and ROC Curve
11. Make predictions
12. Finalize the model
13. Save the trained model as a pickle file

> PyCaret is an AutoML library. It reduces the amount of manual machine-learning code needed for preprocessing, model comparison, tuning, evaluation and deployment.


In [ ]:
import sys
sys.version

## Step 1 – Install PyCaret

Run this cell only if PyCaret is not already installed.

For Python 3.11, the following command normally works well:

```bash
pip install pycaret
```

After installation, restart the Jupyter kernel if required.


In [ ]:
# Uncomment only when PyCaret is not installed
%pip install "pycaret==3.3.2"

## Step 2 – Import Required Libraries

We use:
- **pandas** for loading and inspecting the dataset
- **PyCaret Classification** for the complete AutoML classification workflow


In [ ]:
import pandas as pd

from pycaret.classification import *

## Step 3 – Load the Dataset

The dataset contains customer, income, loan, repayment and credit-related information.

PyCaret will automatically detect:
- Numerical columns
- Categorical columns
- Binary columns

Therefore, we do not need to manually perform one-hot encoding in this notebook.


In [ ]:
df = pd.read_csv("Banking_Loan_Default_Classification.csv")

df.head()

## Step 4 – Check Dataset Shape

This tells us the number of rows and columns available for model building.


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

## Step 5 – Understand the Columns

`df.info()` helps us identify:
- Column names
- Data types
- Number of non-null values

Object columns such as employment type, education level and residence type are categorical variables.


In [ ]:
df.info()

## Step 6 – Check Missing Values

Missing values can affect model performance.

PyCaret can automatically handle many missing-value situations, but it is still important to inspect the dataset before training.


In [ ]:
df.isnull().sum()

## Step 7 – Check Target Class Distribution

For classification, we should check whether the target is balanced.

- Class `0` = No Default
- Class `1` = Default

A moderate imbalance is acceptable, but a very large imbalance may require techniques such as class weighting or resampling.


In [ ]:
print(df["loan_default"].value_counts())

print("\nPercentage distribution:")
print(df["loan_default"].value_counts(normalize=True) * 100)

## Step 8 – Basic Statistical Summary

This gives a quick overview of the numerical variables such as:
- Age
- Income
- Loan amount
- Credit score
- Savings
- Existing debt payments


In [ ]:
df.describe().T

# Step 9 – Initialize PyCaret

`setup()` prepares the complete machine-learning environment.

Important parameters used here:

- `data=df` → Input dataset
- `target="loan_default"` → Column we want to predict
- `train_size=0.80` → 80% training data and 20% test data
- `session_id=42` → Makes results reproducible
- `fold=5` → Uses 5-fold cross-validation
- `fold_strategy="stratifiedkfold"` → Maintains target-class proportion in each fold
- `normalize=True` → Scales numerical features when required
- `fix_imbalance=False` → Keeps the original data distribution for this basic demonstration

PyCaret automatically performs tasks such as:
- Train/test split
- Missing-value handling
- Categorical encoding
- Feature transformation
- Model-ready preprocessing


In [ ]:
classification_setup = setup(
    data=df,
    target="loan_default",
    train_size=0.80,
    session_id=42,
    fold=5,
    fold_strategy="stratifiedkfold",
    normalize=True,
    fix_imbalance=False,
    verbose=True
)

## Step 10 – View Available Classification Models

PyCaret provides several algorithms, including:

- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- K-Nearest Neighbors
- Naive Bayes
- AdaBoost
- Gradient Boosting
- XGBoost, LightGBM and CatBoost when their optional libraries are installed

The command below displays the models available in the current environment.


In [ ]:
models()

# Step 11 – Compare Multiple Models

`compare_models()` trains several algorithms using the same preprocessing and cross-validation process.

PyCaret compares metrics such as:
- Accuracy
- AUC
- Recall
- Precision
- F1 Score
- Kappa
- MCC

Here we optimize for **Accuracy**.

> For loan-default problems, Accuracy should not be considered alone. Recall, Precision, F1 and ROC-AUC should also be reviewed because failing to identify a true defaulter can be important.


In [ ]:
best_model = compare_models(
    sort="Accuracy",
    n_select=1
)

best_model

## Step 12 – View Model Comparison Results

`pull()` retrieves the most recent PyCaret results table so that we can inspect how the algorithms performed.


In [ ]:
comparison_results = pull()
comparison_results

# Step 13 – Inspect the Selected Model

This displays the algorithm selected by PyCaret as the best-performing model according to the chosen optimization metric.


In [ ]:
print(best_model)

# Step 14 – Tune the Best Model

`tune_model()` searches for improved hyperparameter values using cross-validation.

This is similar to performing automated hyperparameter tuning.

The goal is to improve the selected model without manually writing GridSearchCV code.


In [ ]:
tuned_model = tune_model(
    best_model,
    optimize="Accuracy",
    choose_better=True
)

tuned_model

## Step 15 – View Tuning Results

The table below shows cross-validation metrics for the tuned model.

Compare these results with the earlier model-comparison table to see whether tuning improved performance.


In [ ]:
tuning_results = pull()
tuning_results

# Step 16 – Evaluate the Tuned Model

`evaluate_model()` opens PyCaret's interactive evaluation interface.

Depending on the selected model, you can inspect:
- Confusion Matrix
- ROC Curve
- Precision-Recall Curve
- Classification Report
- Learning Curve
- Feature Importance
- Validation Curve


In [ ]:
evaluate_model(tuned_model)

# Step 17 – Confusion Matrix

The confusion matrix shows four important values:

- **True Positive (TP):** Predicted Default and actually Default
- **True Negative (TN):** Predicted No Default and actually No Default
- **False Positive (FP):** Predicted Default but actually No Default
- **False Negative (FN):** Predicted No Default but actually Default

In loan-default prediction, False Negatives can be important because a risky customer may incorrectly be classified as safe.


In [ ]:
plot_model(tuned_model, plot="confusion_matrix")

# Step 18 – ROC Curve and AUC

The ROC curve measures how well the model separates the two classes at different decision thresholds.

### AUC Interpretation
- `0.50` → No useful separation
- `0.60–0.70` → Weak
- `0.70–0.80` → Acceptable
- `0.80–0.90` → Good
- `0.90–1.00` → Very strong

A higher ROC-AUC generally means the model is better at distinguishing default customers from non-default customers.


In [ ]:
plot_model(tuned_model, plot="auc")

# Step 19 – Classification Report

The classification report summarizes:

### Precision
Of all customers predicted as default, how many actually defaulted?

### Recall
Of all customers who actually defaulted, how many were correctly identified?

### F1 Score
A balance between Precision and Recall.

For credit-risk use cases, these metrics are often more informative than Accuracy alone.


In [ ]:
plot_model(tuned_model, plot="class_report")

# Step 20 – Feature Importance

For supported tree-based models, this graph shows which features influenced predictions the most.

Typical influential banking features may include:
- Credit score
- Loan amount
- Debt payments
- Income
- Late-payment history
- Savings balance

> If the selected model does not support native feature importance, PyCaret may not display this plot.


In [ ]:
try:
    plot_model(tuned_model, plot="feature")
except Exception as e:
    print("Feature importance plot is not available for this model.")
    print(e)

# Step 21 – Predict on the Holdout Test Data

`predict_model()` evaluates the tuned model using the test data that PyCaret kept aside during `setup()`.

The output includes:
- Actual target
- Predicted class
- Prediction score/probability


In [ ]:
holdout_predictions = predict_model(tuned_model)

holdout_predictions.head(10)

## Step 22 – Holdout Performance Metrics

After `predict_model()`, `pull()` retrieves the performance metrics calculated on the unseen holdout data.

This is useful because these records were not used to train the model.


In [ ]:
holdout_metrics = pull()
holdout_metrics

# Step 23 – Finalize the Model

`finalize_model()` retrains the selected model using all available data.

Use this only after you are satisfied with the model evaluation.

The finalized model is the model that can be saved and used later in applications such as Streamlit, Flask or FastAPI.


In [ ]:
final_model = finalize_model(tuned_model)

final_model

# Step 24 – Save the Final Model

PyCaret saves both:
- The preprocessing pipeline
- The trained machine-learning model

This is important because future input data must go through the exact same preprocessing steps used during training.

PyCaret automatically creates a `.pkl` file.


In [ ]:
save_model(
    final_model,
    "pycaret_banking_loan_default_model"
)

# Step 25 – Load the Saved Model

The saved model can later be loaded without retraining.

This is useful when the model is deployed in:
- Streamlit
- Flask
- FastAPI
- Batch prediction jobs


In [ ]:
loaded_model = load_model(
    "pycaret_banking_loan_default_model"
)

loaded_model

# Step 26 – Make a Prediction for a New Customer

Create a small dataframe containing the same input columns used during training.

PyCaret automatically applies the saved preprocessing pipeline before prediction.


In [ ]:
new_customer = pd.DataFrame({
    "age": [35],
    "monthly_income_inr": [65000],
    "employment_type": ["Salaried"],
    "years_employed": [7],
    "education_level": ["Graduate"],
    "marital_status": ["Married"],
    "dependents": [2],
    "residence_type": ["Owned"],
    "years_at_address": [5],
    "loan_amount_inr": [500000],
    "loan_term_months": [36],
    "loan_purpose": ["Personal"],
    "existing_loans": [1],
    "monthly_debt_payments_inr": [15000],
    "credit_score": [710],
    "late_payments_last_12m": [1],
    "savings_balance_inr": [200000],
    "account_tenure_years": [6],
    "has_guarantor": ["Yes"]
})

new_customer

In [ ]:
new_customer_prediction = predict_model(
    loaded_model,
    data=new_customer
)

new_customer_prediction

# End-to-End PyCaret Workflow Summary

The complete workflow is:

**CSV Dataset**  
↓  
**Basic EDA**  
↓  
**PyCaret `setup()`**  
↓  
**Automatic Preprocessing**  
↓  
**`compare_models()`**  
↓  
**Select Best Model**  
↓  
**`tune_model()`**  
↓  
**Evaluate Accuracy, Precision, Recall, F1 and ROC-AUC**  
↓  
**Confusion Matrix / ROC Curve**  
↓  
**Holdout Prediction**  
↓  
**`finalize_model()`**  
↓  
**`save_model()`**  
↓  
**Deploy or reuse the `.pkl` model**

## Key Learning Point

With traditional scikit-learn, we normally write separate code for preprocessing, encoding, model training, cross-validation, tuning and evaluation.

**PyCaret automates most of these steps with a small number of commands**, making it useful for:
- Rapid experimentation
- Comparing many ML models
- AutoML demonstrations
- Building baseline models quickly
